# Verify `hist_corpus_qa_1950_1999.jsonl`

Does the merged JSONL row count match the expected count from source files?  
**Expected per collection = QA_files x 3 + CoT_files x 2**

In [ ]:
import json, os
from pathlib import Path
import pandas as pd

PERIOD = "1950_1999"
BASE = Path(r"C:\Users\danielyoon\Dropbox\hist_LLM\data\periods_data") / PERIOD / "posttraining_data"
MERGED_JSONL = BASE / f"hist_corpus_qa_{PERIOD}.jsonl"
GENERATED_DIR = BASE / "synthetic" / "generated"

print(f"Merged JSONL: {MERGED_JSONL}")
print(f"Size: {MERGED_JSONL.stat().st_size / 1e6:.1f} MB")

## 1. Count rows in merged JSONL (QA vs CoT)

In [ ]:
merged_total = 0
merged_qa = 0
merged_cot = 0

with open(MERGED_JSONL, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        conv = json.loads(line)
        merged_total += 1
        if len(conv) >= 2 and "<think>" in conv[1]["content"]:
            merged_cot += 1
        else:
            merged_qa += 1

print(f"Merged JSONL rows: {merged_total:,}")
print(f"  QA:  {merged_qa:,}")
print(f"  CoT: {merged_cot:,}")

## 2. Count source JSON files per collection

Each collection in `generated/` has individual `*_qa_*.json` and `*_cot_*.json` files.  
Expected items: **3 per QA file, 2 per CoT file** (what we requested from the API).

In [ ]:
collections = sorted([d.name for d in GENERATED_DIR.iterdir() if d.is_dir()])
rows = []

for coll in collections:
    coll_dir = GENERATED_DIR / coll
    print(f"{coll}...", end=" ", flush=True)

    qa_files = cot_files = 0
    for entry in os.scandir(coll_dir):
        if not entry.name.endswith(".json"):
            continue
        if "_qa_" in entry.name:
            qa_files += 1
        elif "_cot_" in entry.name:
            cot_files += 1

    expected = qa_files * 3 + cot_files * 2
    rows.append({"collection": coll,
                 "qa_files": qa_files, "cot_files": cot_files,
                 "expected_total": expected})
    print(f"{qa_files:,} QA x3 + {cot_files:,} CoT x2 = {expected:,}")

print("\nDone.")

## 3. Verification

In [ ]:
df = pd.DataFrame(rows)
expected_grand_total = df["expected_total"].sum()
diff = merged_total - expected_grand_total

print("=" * 65)
print(f"  Merged JSONL rows:       {merged_total:>10,}  (QA: {merged_qa:,}  CoT: {merged_cot:,})")
print(f"  Expected from sources:   {expected_grand_total:>10,}  (QA_files x3 + CoT_files x2)")
print(f"  Difference:              {diff:>+10,}  ({abs(diff)/expected_grand_total*100:.1f}%)")
print(f"  Status:                  {'MATCH' if diff == 0 else 'MISMATCH'}")
print("=" * 65)
print()

# Flag anomalies
print("Anomalies:")
for _, row in df.iterrows():
    # QA and CoT file counts should be equal (1 QA + 1 CoT per chunk)
    if row["qa_files"] != row["cot_files"]:
        print(f"  {row['collection']}: {row['qa_files']:,} QA files != {row['cot_files']:,} CoT files")
print()

# Per-collection table
df

## 4. Explain the difference

Two factors:
1. **`nyt_filtered` has ~20K QA files** instead of ~10K — duplicate QA from both `run.py` and `run_direct.py`
2. **API returns variable items per request** — sometimes more or fewer than the 3 QA / 2 CoT requested

In [ ]:
# What would the expected be if nyt_filtered had 10K QA files (no duplicates)?
nyt_row = df[df["collection"] == "nyt_filtered"].iloc[0]
extra_qa_files = nyt_row["qa_files"] - 10000
corrected_expected = expected_grand_total - extra_qa_files * 3
api_variance = merged_total - corrected_expected

print(f"nyt_filtered extra QA files:     {extra_qa_files:,} (duplicate from run.py + run_direct.py)")
print(f"Extra conversations from dupes:  ~{extra_qa_files * 3:,}")
print()
print(f"Corrected expected (no dupes):   {corrected_expected:,}")
print(f"Merged actual:                   {merged_total:,}")
print(f"Remaining diff (API variance):   {api_variance:+,} ({abs(api_variance)/corrected_expected*100:.1f}%)")